# 10 — Full POI Enrichment (All 20,000 Outlets)

Fixes the 4.5% coverage problem in `03_poi_enrichment.ipynb`.

**Root cause of 902/20,000:** old notebook built the bounding box from the 914-row submission template.

**Output:** `data/gold/outlet_poi_features_full.csv` (does NOT overwrite the existing sparse file)

**Changes vs 03:**
- Full Sri Lanka bbox for Overpass (cached as `poi_overpass_full_<category>.json`)
- All 20,000 outlets processed (distributor-median coord imputation for 240 invalid)
- Gaussian-decay weighted scores (σ=0.75 km urban)
- Defaults: `dist = 4 km`, `count = 0` — never NaN

## 1. Setup

In [1]:
from __future__ import annotations
import json, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import requests
from sklearn.neighbors import BallTree
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
try:
    pd.options.future.infer_string = False
except AttributeError:
    pass

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'Notebooks' else Path.cwd()
SILVER_DIR   = PROJECT_ROOT / 'data' / 'silver'
BRONZE_DIR   = PROJECT_ROOT / 'data' / 'bronze'
GOLD_DIR     = PROJECT_ROOT / 'data' / 'gold'
PBF_PATH     = PROJECT_ROOT / 'data' / 'sri-lanka-latest.osm.pbf'

OVERPASS_URL  = 'https://overpass-api.de/api/interpreter'
GEOFABRIK_URL = 'https://download.geofabrik.de/asia/sri-lanka-latest.osm.pbf'

# Output — NEW file, does not overwrite sparse 03 output
OUTPUT_POI_CSV  = GOLD_DIR / 'outlet_poi_features_full.csv'
OUTPUT_CHART    = GOLD_DIR / 'poi_coverage_summary_full.png'
CACHE_PREFIX    = 'poi_overpass_full_'   # separate cache from 03's files

LK_SOUTH, LK_NORTH = 5.85,  9.90
LK_WEST,  LK_EAST  = 79.65, 81.90
LK_BBOX = f'{LK_SOUTH},{LK_WEST},{LK_NORTH},{LK_EAST}'

RADIUS_KM   = [0.25, 0.5, 1.0, 2.0]
EARTH_KM    = 6371.0088
SIGMA_URBAN = 0.75

POI_FILTERS = {
    'education'     : [('amenity','school'),('amenity','college'),('amenity','university'),('amenity','kindergarten')],
    'transport'     : [('highway','bus_stop'),('amenity','bus_station'),('railway','station'),('railway','halt')],
    'market_retail' : [('shop','supermarket'),('shop','convenience'),('shop','mall'),('amenity','marketplace')],
    'healthcare'    : [('amenity','hospital'),('amenity','clinic'),('amenity','pharmacy')],
    'food_service'  : [('amenity','restaurant'),('amenity','cafe'),('amenity','fast_food'),('shop','bakery')],
    'office_finance': [('amenity','bank'),('amenity','atm'),('office','company'),('office','government')],
    'religious'     : [('amenity','place_of_worship')],
    'tourism_hotel' : [('tourism','hotel'),('tourism','guest_house'),('tourism','attraction')],
    'parks_leisure' : [('leisure','park'),('leisure','playground'),('leisure','sports_centre')],
}
DEMAND_WEIGHTS = {
    'market_retail':0.25,'transport':0.20,'food_service':0.15,'education':0.12,
    'office_finance':0.10,'healthcare':0.08,'tourism_hotel':0.05,'religious':0.03,'parks_leisure':0.02,
}
print(f'Root: {PROJECT_ROOT}')
print(f'Output: {OUTPUT_POI_CSV.relative_to(PROJECT_ROOT)}')
print(f'LK bbox: {LK_BBOX}')

ModuleNotFoundError: No module named 'matplotlib'

## 2. Load All Outlets + Coordinate Imputation

In [ ]:
outlets    = pd.read_csv(SILVER_DIR / 'outlet_master.csv')
coords_raw = pd.read_csv(SILVER_DIR / 'outlet_coordinates.csv')
outlets['Outlet_ID']    = outlets['Outlet_ID'].astype(str)
coords_raw['Outlet_ID'] = coords_raw['Outlet_ID'].astype(str)
coords_raw['Latitude']  = pd.to_numeric(coords_raw['Latitude'],  errors='coerce')
coords_raw['Longitude'] = pd.to_numeric(coords_raw['Longitude'], errors='coerce')
coords_raw['valid']     = coords_raw['Latitude'].between(5.5,10.2) & coords_raw['Longitude'].between(79.0,82.1)
coords_raw.loc[~coords_raw['valid'], ['Latitude','Longitude']] = np.nan

monthly = pd.read_csv(GOLD_DIR / 'outlet_monthly_sales.csv')
monthly['Outlet_ID']      = monthly['Outlet_ID'].astype(str)
monthly['Distributor_ID'] = monthly['Distributor_ID'].astype(str)
dom_dist = (
    monthly.groupby(['Outlet_ID','Distributor_ID']).size().rename('n').reset_index()
    .sort_values(['Outlet_ID','n'],ascending=[True,False])
    .drop_duplicates('Outlet_ID')[['Outlet_ID','Distributor_ID']]
)

geo = outlets[['Outlet_ID']].merge(coords_raw[['Outlet_ID','Latitude','Longitude','valid']], on='Outlet_ID', how='left')
geo = geo.merge(dom_dist, on='Outlet_ID', how='left')
geo['has_valid_coordinates'] = geo['valid'].fillna(False).astype(int)
for col in ['Latitude','Longitude']:
    geo[col] = geo[col].fillna(geo.groupby('Distributor_ID')[col].transform('median'))
    geo[col] = geo[col].fillna(geo[col].median())

all_outlets = geo[['Outlet_ID','Latitude','Longitude','has_valid_coordinates']].copy()
print(f'Total outlets:   {len(all_outlets):,}')
print(f'Valid coords:    {all_outlets["has_valid_coordinates"].sum():,}')
print(f'Imputed:         {(all_outlets["has_valid_coordinates"]==0).sum():,}')

## 3. Acquire POI Data

Geofabrik PBF → pyrosm (if available), else full-country Overpass. Caches as `poi_overpass_full_<cat>.json`.

In [ ]:
poi_df = None
source_used = 'none'

# Option A: Geofabrik PBF via pyrosm
if PBF_PATH.exists():
    try:
        import pyrosm
        print(f'PBF found. Parsing with pyrosm...')
        osm = pyrosm.OSM(str(PBF_PATH))
        rows = []
        for category, filters in POI_FILTERS.items():
            for key, val in filters:
                try:
                    pois = osm.get_pois(custom_filter={key: [val]})
                    if pois is None or pois.empty: continue
                    pois = pois.copy()
                    pois['poi_category'] = category
                    pois['lat'] = pois.geometry.centroid.y
                    pois['lon'] = pois.geometry.centroid.x
                    rows.append(pois[['poi_category','lat','lon']].rename(columns={'lat':'Latitude','lon':'Longitude'}))
                except Exception:
                    pass
        if rows:
            poi_df = pd.concat(rows, ignore_index=True).dropna(subset=['Latitude','Longitude'])
            poi_df = poi_df[poi_df['Latitude'].between(5.5,10.2) & poi_df['Longitude'].between(79.0,82.1)]
            source_used = 'geofabrik_pyrosm'
            print(f'pyrosm: {len(poi_df):,} POIs')
    except ImportError:
        print('pyrosm not installed — falling back to Overpass')
    except Exception as e:
        print(f'pyrosm failed ({e}) — falling back to Overpass')

# Option B: Download PBF if not present (SSL bypass for macOS cert issue)
if poi_df is None and not PBF_PATH.exists():
    print(f'Attempting Geofabrik download (~136 MB) ...')
    try:
        # Use requests with verify=False to bypass macOS SSL cert issue
        import shutil
        with requests.get(GEOFABRIK_URL, stream=True, verify=False, timeout=600) as resp:
            resp.raise_for_status()
            with open(PBF_PATH, 'wb') as f:
                shutil.copyfileobj(resp.raw, f)
        print(f'Downloaded: {PBF_PATH.stat().st_size/1e6:.1f} MB — re-run cell to parse with pyrosm')
        # Attempt immediate parse if pyrosm available
        try:
            import pyrosm
            poi_df = None  # will be caught by Option A on re-run; skip for now
        except ImportError:
            pass
    except Exception as e:
        print(f'Download failed ({e}) — using Overpass')

# Option C: Full-country Overpass (separate cache from 03)
if poi_df is None:
    print(f'Overpass API | bbox: {LK_BBOX}')

    def _q(category, bbox):
        clauses = []
        for key, val in POI_FILTERS[category]:
            clauses.append(f'node["{key}"="{val}"]({bbox});')
            clauses.append(f'way["{key}"="{val}"]({bbox});')
        return '[out:json][timeout:180];\n(\n  ' + '\n  '.join(clauses) + '\n);\nout center;'

    def _fetch(category):
        cache = BRONZE_DIR / f'{CACHE_PREFIX}{category}.json'
        if cache.exists():
            data = json.loads(cache.read_text('utf-8'))
            print(f'  cache: {category} ({len(data.get("elements",[]))} el)')
        else:
            print(f'  fetch: {category}...')
            r = requests.post(OVERPASS_URL, data={'data': _q(category, LK_BBOX)},
                              headers={'User-Agent':'DataStorm2026-NB10/1.0'}, timeout=240)
            if r.status_code != 200:
                print(f'  HTTP {r.status_code}')
                return pd.DataFrame(columns=['poi_category','Latitude','Longitude'])
            data = r.json()
            cache.write_text(json.dumps(data), 'utf-8')
            time.sleep(3)
        rows = []
        for el in data.get('elements', []):
            lat = el.get('lat') or el.get('center', {}).get('lat')
            lon = el.get('lon') or el.get('center', {}).get('lon')
            if lat and lon:
                rows.append({'poi_category': category, 'Latitude': float(lat), 'Longitude': float(lon)})
        return pd.DataFrame(rows)

    poi_df = pd.concat([_fetch(c) for c in POI_FILTERS], ignore_index=True)
    poi_df = poi_df.dropna(subset=['Latitude','Longitude'])
    poi_df = poi_df[poi_df['Latitude'].between(5.5,10.2) & poi_df['Longitude'].between(79.0,82.1)]
    source_used = 'overpass_lk_bbox'

poi_df = poi_df.drop_duplicates(['poi_category','Latitude','Longitude']).reset_index(drop=True)
print(f'\nSource: {source_used} | Total POIs: {len(poi_df):,}')
print(poi_df['poi_category'].value_counts())

## 4. Build Outlet-Level Features

In [ ]:
outlet_rad = np.radians(all_outlets[['Latitude','Longitude']].to_numpy())
n = len(all_outlets)
poi_feat = pd.DataFrame({'Outlet_ID': all_outlets['Outlet_ID'].values})
total_1km = np.zeros(n, dtype=float)
total_2km = np.zeros(n, dtype=float)
demand_accum = np.zeros(n, dtype=float)
MAX_DIST_DEFAULT = RADIUS_KM[-1] * 2

for category in POI_FILTERS:
    cpoi   = poi_df[poi_df['poi_category'] == category]
    weight = DEMAND_WEIGHTS.get(category, 0.02)
    if cpoi.empty:
        for r in RADIUS_KM:
            poi_feat[f'poi_{category}_count_{str(r).replace(".","p")}km'] = 0
        poi_feat[f'poi_{category}_nearest_km']    = MAX_DIST_DEFAULT
        poi_feat[f'poi_{category}_gauss_score']   = 0.0
        print(f'  {category}: 0 POIs')
        continue
    ptree = BallTree(np.radians(cpoi[['Latitude','Longitude']].to_numpy()), metric='haversine')
    for r in RADIUS_KM:
        cnts = ptree.query_radius(outlet_rad, r=r/EARTH_KM, count_only=True).astype(float)
        poi_feat[f'poi_{category}_count_{str(r).replace(".","p")}km'] = cnts
        if r == 1.0: total_1km += cnts
        if r == 2.0: total_2km += cnts
    dist, _ = ptree.query(outlet_rad, k=1)
    near_km = np.minimum(dist[:,0]*EARTH_KM, MAX_DIST_DEFAULT)
    poi_feat[f'poi_{category}_nearest_km']  = near_km
    gauss = np.exp(-0.5*(near_km/SIGMA_URBAN)**2)
    poi_feat[f'poi_{category}_gauss_score'] = gauss
    demand_accum += weight * gauss
    print(f'  {category}: {len(cpoi):,} POIs | median nearest {np.median(near_km):.3f} km')

poi_feat['poi_total_count_1km']  = total_1km.astype(int)
poi_feat['poi_total_count_2km']  = total_2km.astype(int)
poi_feat['poi_demand_score_raw'] = demand_accum
poi_feat['poi_demand_score']     = pd.Series(demand_accum).rank(pct=True).clip(0,1).values

print(f'\nFeature matrix: {poi_feat.shape}')
print(f'Coverage: {(poi_feat["poi_demand_score"]>0).sum():,} / {len(poi_feat):,}')
print(poi_feat[['poi_total_count_1km','poi_total_count_2km','poi_demand_score']].describe().round(3))

## 5. Validate & Save

In [ ]:
checks = {
    'Row count == 20 000'    : len(poi_feat) == 20_000,
    'No null Outlet_ID'      : poi_feat['Outlet_ID'].notna().all(),
    'No NaN demand_score'    : poi_feat['poi_demand_score'].notna().all(),
    'No NaN nearest cols'    : not poi_feat.filter(like='_nearest_km').isna().any().any(),
    'Coverage > old 902'     : (poi_feat['poi_demand_score'] > 0).sum() > 902,
    'No duplicate Outlet_ID' : poi_feat['Outlet_ID'].duplicated().sum() == 0,
    'Does NOT overwrite old' : not (GOLD_DIR / 'outlet_poi_features.csv') == OUTPUT_POI_CSV,
}
for k, v in checks.items(): print(f"{'✓' if v else '✗'} {k}")

poi_feat.to_csv(OUTPUT_POI_CSV, index=False)
print(f'\nSaved: {OUTPUT_POI_CSV.relative_to(PROJECT_ROOT)}')
print(f'Old sparse coverage: 902  →  New full coverage: {(poi_feat["poi_demand_score"]>0).sum():,}')
print('Old file intact:     data/gold/outlet_poi_features.csv (unchanged)')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
poi_feat['poi_total_count_1km'].clip(0,50).hist(bins=40, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('POI count within 1 km')
poi_feat['poi_demand_score'].hist(bins=40, ax=axes[1], color='tomato', edgecolor='white')
axes[1].set_title('poi_demand_score (full coverage)')
poi_df['poi_category'].value_counts().sort_values().plot(kind='barh', ax=axes[2], color='seagreen')
axes[2].set_title('POI count by category')
plt.tight_layout()
plt.savefig(OUTPUT_CHART, dpi=150, bbox_inches='tight')
plt.show()
print(f'Chart: {OUTPUT_CHART.relative_to(PROJECT_ROOT)}')